Simulated Backtest

In [15]:
import yfinance as yf
import pandas as pd

BIG_SMA = 100
SMALL_SMA = 50
STOP_LOSS_PERCENTAGE = 0.02
RR_RATIO = 2
ACCOUNT_SIZE = 10000
PERCENTAGE_INVESTED = 1.0

df = yf.download("SPY", period="2y", interval="4h", multi_level_index=False)

[*********************100%***********************]  1 of 1 completed


In [16]:
df["SMA100"] = df["Close"].rolling(BIG_SMA).mean()
df["SMA50"] = df["Close"].rolling(SMALL_SMA).mean()

df["Change"] = df["Close"] - df["Close"].shift(1)
df["Increase"] = df["Change"].clip(lower=0)
df["Decrease"] = (df["Change"].clip(upper=0)).abs()

df["RSI"] = df["Increase"].rolling(14).mean() / df["Decrease"].rolling(14).mean()
df.dropna(inplace=True)

In [17]:
in_position = False
trades = []

for row in df.itertuples():
    if row.RSI < 30:
        if not in_position:
            print(f"Buy at {row.Close}")
            in_position = True
            trades.append({"Entry Price": row.Close, "Stop Loss": row.Close * (1 - STOP_LOSS_PERCENTAGE), "Take Profit": row.Close * (1 + STOP_LOSS_PERCENTAGE * RR_RATIO)})
    elif row.RSI > 70:
        if in_position:
            print(f"Sell at {row.Close}")
            in_position = False
            trades[-1]["Exit Price"] = row.Close

    if in_position:
        if row.Close <= trades[-1]["Stop Loss"]:
            print(f"Stop Loss hit at {row.Close}")
            in_position = False
            trades[-1]["Exit Price"] = row.Close
        elif row.Close >= trades[-1]["Take Profit"]:
            print(f"Take Profit hit at {row.Close}")
            in_position = False
            trades[-1]["Exit Price"] = row.Close

trades_df = pd.DataFrame(trades)
trades_df["PnL"] = trades_df["Exit Price"] - trades_df["Entry Price"]
trades_df["PnL %"] = trades_df["PnL"] / trades_df["Entry Price"] * 100

print(f"Total PnL: {trades_df['PnL'].sum():.2f}")
print(f"Total PnL %: {trades_df['PnL %'].sum():.2f}%")

Buy at 557.1199951171875
Stop Loss hit at 540.8300170898438
Buy at 540.4199829101562
Take Profit hit at 562.760009765625
Buy at 561.9600219726562
Take Profit hit at 584.6199951171875
Buy at 584.5700073242188
Stop Loss hit at 571.176513671875
Buy at 568.5700073242188
Take Profit hit at 594.8499755859375
Buy at 595.5499877929688
Stop Loss hit at 582.3599853515625
Buy at 584.5700073242188
Take Profit hit at 607.9849853515625
Buy at 609.7100219726562
Stop Loss hit at 596.125
Buy at 599.4099731445312
Stop Loss hit at 585.0399780273438
Buy at 584.719970703125
Stop Loss hit at 572.530029296875
Buy at 572.8200073242188
Stop Loss hit at 560.75
Buy at 552.2000122070312
Take Profit hit at 575.3499755859375
Buy at 568.9500122070312
Stop Loss hit at 555.3900146484375
Buy at 555.8200073242188
Stop Loss hit at 542.260009765625
Buy at 537.6400146484375
Stop Loss hit at 511.1600036621094
Buy at 505.510009765625
Take Profit hit at 542.8400268554688
Buy at 519.3200073242188
Take Profit hit at 544.9799804